Cell 1: Import the Libraries


##Enviroment

In [ ]:
pip install langchain-google-genai langsmith pypdf

In [ ]:
import getpass
import os

# Securely set API Keys using getpass
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google API Key:")
# Enable LangSmith Tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://eu.api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = "Rag"

**Part 2: Load the Data**

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, BSHTMLLoader, WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
import bs4

In [ ]:
print("PHASE 1: Indexing Pipeline")

docs = []
# Load All PDFs
print("Loading PDFs...")
pdf_files = [f"./data/{file}" for file in os.listdir("./data") if file.endswith('.pdf')]

print(f"Found {len(pdf_files)} PDFs.")
for file_path in pdf_files:
    loader = PyPDFLoader(file_path)
    docs.extend(loader.load())

Scraping live web data...

In [ ]:
print("Scraping live web data...")
url = "https://en.wikipedia.org/wiki/Prompt_injection"
web_loader = WebBaseLoader(
    web_paths=[url],
    bs_kwargs=dict(parse_only=bs4.SoupStrainer("div", id="mw-content-text"))
)
docs.extend(web_loader.load())

In [ ]:
print(f"Total separate documents loaded: {len(docs)}")

SPLIT: Break documents into smaller chunks

In [ ]:
print(f"\nTotal documents loaded: {len(docs)}. Chunking text...")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(docs)
print(f"Created {len(chunks)} text chunks.")
if len(chunks) > 0:
    print(f"Length of the first chunk: {len(chunks[0].page_content)} characters")

STORE: Insert embeddings into Chroma vector store

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Initialize the Embeddings Model using the current active model
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

print("Embedding and storing in Chroma...")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_gemini_db"
)

In [ ]:
# 1. Get the total number of chunks securely stored in the DB
db_chunk_count = vectorstore._collection.count()
print(f"Total chunks stored in ChromaDB: {db_chunk_count}")

In [ ]:
# if db_chunk_count > 0:
#     # 2. Fetch one specific chunk directly from the database
#     # The .get() method pulls the raw data arrays (ids, documents, metadatas)
#     sample_data = vectorstore.get(limit=1)

#     sample_id = sample_data['ids'][0]
#     sample_text = sample_data['documents'][0]
#     sample_metadata = sample_data['metadatas'][0]

#     chunk_length = len(sample_text)

#     # 3. Print the results
#     print(f"\n--- Inspection for Vector ID: {sample_id} ---")
#     print(f"Length of this chunk: {chunk_length} characters")
#     print(f"Source: {sample_metadata.get('source', 'Unknown')}")
#     print("-" * 40)
#     print(sample_text)
#     print("-" * 40)
# else:
#     print("The database is currently empty.")

**Part 3 : Retrieval**

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

print("Retrieval Pipeline")
# 1. RETRIEVE: Set up semantic search over the Chroma vector store
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [ ]:
docs = retriever.invoke("What is prompt injection?")

In [ ]:
len(docs)

**Part 4 : Generation**

In [ ]:
print("Initializing Google GenAI Models")
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

template = """
You are a backend engineering assistant.
Use ONLY the following pieces of retrieved context to answer the question.
If the answer is not in the context, say "I do not have that information.""

Context = {context}

Question = {question}

Answer: """

prompt = PromptTemplate.from_template(template)

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Rag is REady.")

**Execution & Tracing**

In [ ]:
response = rag_chain.invoke("What is the endpoint for the modern facturation API v2 and what is the required payload?")

In [ ]:
print(f"Response : {response}")

In [ ]:
rag_chain.invoke("What are the core differences in data privacy between Federated Learning and traditional centralized machine learning training models?")

In [ ]:
rag_chain.invoke("If an LLM begins disregarding its system prompt during an internal support session, what specific actions must the engineering team take, and how do we handle memory leaks in the routing engine?")

In [ ]:
rag_chain.invoke("What is the general definition of prompt injection, and according to our internal security protocols, what specific actions must the team take if an LLM begins outputting unauthorized commands?")

In [ ]:
rag_chain.invoke("what is 1 + 1 ?")